# SPARC Pipeline Demo

This notebook demonstrates the basic usage of the SPARC (Spectral Pattern Analysis for ROI Classification) pipeline. 

**Prerequisites:**
1. Ensure you have downloaded the SAM model weights (`sam_vit_h_4b8939.pth`).
2. Ensure you have the required external repositories installed (`asdf`, `pdr`, etc.).
3. Set the `IOF_DATA_PATH` variable below to point to your data directory.

In [ ]:
import sys
import logging
from pathlib import Path

# Import SPARC
from src.sparc import Sparc, export_spectra_csv, configure_logging

# Configure logging to see detailed progress
configure_logging(verbose=True)

In [ ]:
# --- CONFIGURATION ---

# Path to the downloaded Segment Anything Model weights
SAM_MODEL_PATH = "./models/sam_vit_h_4b8939.pth"

# Path to your Marslab/ZCAM data directory
IOF_DATA_PATH = "/path/to/your/data/directory"

# Sequence ID and Observation Index (specific to your dataset)
SEQ_ID = None  # Or specific sequence ID string
OBS_IX = 0

## 1. Initialize Pipeline

We initialize the `Sparc` class with the path to the SAM model. We enable GPU acceleration and threading for performance.

In [ ]:
sparc = Sparc(
    sam_model_path=SAM_MODEL_PATH,
    use_gpu=True,       # Will fall back to CPU if CUDA is not available
    use_threading=True, # Enables parallel ROI extraction
    verbose=True        # Print debug stats
)

## 2. Run Processing Steps

We can chain the methods together to run the full pipeline. 

* **Load:** Reads IOF data and aligns stereo cameras.
* **Preprocess:** Masks shadows/sky and converts to R*.
* **Segment:** Uses SAM to find semantic regions.
* **Extract ROIs:** Sub-clusters segments to find homogeneous spectral regions.
* **Analyze:** Detects outliers and clusters spectra.
* **Select:** Picks the best representative ROI for each cluster.

In [ ]:
(
    sparc
    .load(
        iof_path=IOF_DATA_PATH, 
        seq_id=SEQ_ID, 
        obs_ix=OBS_IX
    )
    .preprocess(
        apply_r_star=True
    )
    .segment(
        points_per_side=32,   # Higher = more detailed segmentation
        pred_iou_thresh=0.88
    )
    .extract_rois(
        area_threshold=50,       # Minimum ROI size in pixels
        min_cluster_area=500,    # Min segment size to attempt sub-clustering
        min_clean_area=4000      # Min size after cleaning artifacts
    )
    .analyze(
        contamination=0.1,    # Estimated % of outliers in data
        max_components=None   # Auto-detect number of spectral clusters
    )
    .select()                 # Final heuristic selection
)

## 3. Visualization

Plot the summary of the run, showing the RGB context, the segmentation map, the final selected ROIs, and their corresponding spectra.

In [ ]:
sparc.plot(figsize=(16, 12))

## 4. Export Results

Save the spectral data to a CSV file compatible with Marslab analysis tools.

In [ ]:
output_dir = Path("output")
output_dir.mkdir(exist_ok=True)

csv_path = output_dir / "demo_spectra.csv"
export_spectra_csv(sparc.result, str(csv_path))

print(f"Results exported to {csv_path}")

## 5. Accessing Data Programmatically

You can access the raw data directly from the result object.

In [ ]:
result = sparc.result

print(f"Scene ID: {result.scene_id}")
print(f"Number of final ROIs: {len(result.final_rois)}")
print(f"Number of spectral clusters: {result.n_clusters}")

# Example: Print coordinates of the first ROI
if len(result.final_rois) > 0:
    x, y, w, h = result.final_rois[0]
    print(f"First ROI: x={x}, y={y}, w={w}, h={h}")